In [2]:
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
RANDOM_STATE = 42

In [3]:
mat = pd.read_csv('student-mat.csv', sep=';')
mat.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [4]:
def risk_level(G3):
    if G3 <= 9:
        return "High"
    elif G3 <= 14:
        return "Medium"
    else:
        return "Low"

mat["Risk_level"] = mat["G3"].apply(risk_level)

mat.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,Risk_level
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,High
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,High
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,Medium
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,Low
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,Medium


In [5]:
X = mat.drop(["G2", "G3", "Risk_level"], axis=1)
y = mat["Risk_level"]

In [6]:
encoder = LabelEncoder()

binary_features = [
    "school",
    "sex",
    "address",
    "famsize",
    "Pstatus",
    "schoolsup",
    "famsup",
    "paid",
    "activities",
    "nursery",
    "higher",
    "internet",
    "romantic"
]

for col in binary_features:
    X[col] = encoder.fit_transform(X[col])

binary_encoded_features = X[binary_features]

In [7]:
categorical_features = [
    "Mjob",
    "Fjob",
    "reason",
    "guardian"
]

ohe = OneHotEncoder(
    drop="first",
    sparse_output=False
)

categorical_encoded = ohe.fit_transform(
    X[categorical_features]
)

categorical_encoded_features = pd.DataFrame(
    categorical_encoded,
    columns=ohe.get_feature_names_out(categorical_features),
    index=X.index
)

In [8]:
numeric_features = [
    "age",
    "Medu",
    "Fedu",
    "traveltime",
    "studytime",
    "failures",
    "famrel",
    "freetime",
    "goout",
    "Dalc",
    "Walc",
    "health",
    "absences",
    "G1"
]

scaler = StandardScaler()

scaled = scaler.fit_transform(
    X[numeric_features]
)

scaled_features = pd.DataFrame(
    scaled,
    columns=numeric_features,
    index=X.index
)

In [9]:
X_processed = pd.concat(
    [
        binary_encoded_features,
        categorical_encoded_features,
        scaled_features
    ],
    axis=1
)

X_processed.head()

,school,sex,address,famsize,Pstatus,schoolsup,famsup,paid,activities,nursery,...,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1
0,0,0,1,0,0,1,0,0,0,1,...,-0.042286,-0.449944,0.062194,-0.236010,0.801479,-0.540699,-1.003789,-0.399289,0.036424,-1.782467
1,0,0,1,0,1,0,1,0,0,0,...,-0.042286,-0.449944,1.178860,-0.236010,-0.097908,-0.540699,-1.003789,-0.399289,-0.213796,-1.782467
2,0,0,1,1,1,1,0,1,0,1,...,-0.042286,3.589323,0.062194,-0.236010,-0.997295,0.583385,0.551100,-0.399289,0.536865,-1.179147
3,0,0,1,0,1,0,1,1,1,1,...,1.150779,-0.449944,-1.054472,-1.238419,-0.997295,-0.540699,-1.003789,1.041070,-0.464016,1.234133
4,0,0,1,0,1,0,1,1,0,1,...,-0.042286,-0.449944,0.062194,-0.236010,-0.997295,-0.540699,-0.226345,1.041070,-0.213796,-1.480807


In [10]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_processed,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (276, 40)
Validation: (59, 40)
Testing: (60, 40)


In [11]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(grid_search.best_score_)

Best Parameters:
{'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}

Best Cross Validation Accuracy:
0.7536363636363637


In [12]:
y_test_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, average="weighted")
recall = recall_score(y_test, y_test_pred, average="weighted")
f1 = f1_score(y_test, y_test_pred, average="weighted")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.7666666666666667
Precision: 0.8
Recall: 0.7666666666666667
F1 Score: 0.7604317165462676


In [13]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

        High       0.85      0.55      0.67        20
         Low       1.00      0.73      0.84        11
      Medium       0.69      0.93      0.79        29

    accuracy                           0.77        60
   macro avg       0.85      0.74      0.77        60
weighted avg       0.80      0.77      0.76        60



In [14]:
print(confusion_matrix(y_test, y_test_pred))

[[11  0  9]
 [ 0  8  3]
 [ 2  0 27]]


In [15]:
comparison = pd.DataFrame({
    "Model": ["Sprint 1 Random Forest", "Sprint 2 Tuned Random Forest"],
    "Accuracy": [0.593220, accuracy]
})

comparison

,Model,Accuracy
0,Sprint 1 Random Forest,0.593220
1,Sprint 2 Tuned Random Forest,0.766667


In [16]:
joblib.dump(best_model, "student_risk_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [17]:
y_val_pred = best_model.predict(X_val)

validation_accuracy = accuracy_score(y_val, y_val_pred)

print("Validation Accuracy:", validation_accuracy)

Validation Accuracy: 0.7796610169491526
